<a href="https://colab.research.google.com/github/ssk-algoverse/sae-binding/blob/main/Activation_Patching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformer_lens

In [2]:
import torch

In [3]:
from huggingface_hub import hf_hub_download

REPO_ID = "sebastianhoenig/2L_1H_Entity_Binding"
FILENAME = "2L_1H_Attn_Only.pth"

weights_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


2L_1H_Attn_Only.pth:   0%|          | 0.00/2.63M [00:00<?, ?B/s]

In [4]:
### Model

from transformer_lens import HookedTransformer, HookedTransformerConfig

E = 100 # num entities
A = 100 # num attributes
T = 10 # num types/relations
SEP = E+A+T # as seperator between relations
Q = E+A+T+1 # question token
PAD = E+A+T+2
D_VOCAB = E+A+T+3
IGNORE_INDEX = -100

cfg = HookedTransformerConfig(
    n_layers=2,
    n_heads=1,
    d_model=256,
    d_head=256,
    d_mlp=1024,
    n_ctx=64,
    d_vocab=D_VOCAB,
    act_fn="gelu",
    attn_only=True,
    normalization_type="LN",
)
model = HookedTransformer(cfg)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Load the model
# Create a new model instance with the same configuration
pretrained_weights = torch.load(weights_path, map_location=device, weights_only=True)
model.load_state_dict(pretrained_weights)

print("Model loaded successfully.")

Moving model to device:  cuda
Model loaded successfully.


In [5]:
import numpy as np
import pandas as pd
import torch
import ast
from torch.utils.data import Dataset, DataLoader


In [13]:
id_mapping_df = pd.read_csv('id_mapping.csv')
id_to_entity = dict(zip(id_mapping_df['id'], id_mapping_df['name']))


# Add mappings for special tokens
id_to_entity[210] = ','
id_to_entity[211] = '?'

In [37]:
class EntityBindingDataset(Dataset):
    def __init__(self, dataframe, parse_tokens_if_str=True):
        self.df = dataframe.reset_index(drop=True)
        self.parse_tokens_if_str = parse_tokens_if_str

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq = row["tokens"]
        tokens = torch.tensor(seq, dtype=torch.long)
        label  = torch.tensor(int(row["label"]), dtype=torch.long)
        return tokens, label

test_df = pd.read_csv('test_df.csv', converters={"tokens": ast.literal_eval})
test_dataset = EntityBindingDataset(test_df)

In [38]:
from transformer_lens import patching

In [39]:
clean_tokens, label = test_dataset[0]

In [40]:
print("Clean tokens and their mapped entities:")
for token_id in clean_tokens:
    entity_name = id_to_entity.get(token_id.item(), f"Unknown Token {token_id.item()}")
    print(f"Token {token_id.item()} ({entity_name})")

print(f"\nLabel: {label.item()} ({id_to_entity.get(label.item(), f'Unknown Token {label.item()}')})")

Clean tokens and their mapped entities:
Token 94 (Bryan)
Token 201 (was born in)
Token 118 (Istanbul)
Token 210 (,)
Token 43 (Keith)
Token 208 (moved to)
Token 105 (Manila)
Token 210 (,)
Token 18 (Lisa)
Token 200 (lives in)
Token 155 (Philadelphia)
Token 210 (,)
Token 95 (Patrick)
Token 204 (studied in)
Token 144 (Riyadh)
Token 210 (,)
Token 48 (Cynthia)
Token 202 (works in)
Token 106 (Shanghai)
Token 210 (,)
Token 37 (Christine)
Token 205 (married in)
Token 111 (Sao Paulo)
Token 210 (,)
Token 54 (Deborah)
Token 206 (visited)
Token 169 (Cape Town)
Token 210 (,)
Token 80 (Anna)
Token 207 (loves)
Token 154 (Miami)
Token 210 (,)
Token 208 (moved to)
Token 43 (Keith)
Token 211 (?)

Label: 105 (Manila)


In [41]:
# entity and entity_type exists in the context, but not the binding
corrupt_token_1 = clean_tokens.clone()
corrupt_token_1[1], corrupt_token_1[5] = corrupt_token_1[5], corrupt_token_1[1]

print("Entity and entity_type exists in the context, but not the binding:")
for token_id in corrupt_token_1:
    entity_name = id_to_entity.get(token_id.item(), f"Unknown Token {token_id.item()}")
    print(f"Token {token_id.item()} ({entity_name})")

print(f"\nLabel: {label.item()} ({id_to_entity.get(label.item(), f'Unknown Token {label.item()}')})")

Entity and entity_type exists in the context, but not the binding:
Token 94 (Bryan)
Token 208 (moved to)
Token 118 (Istanbul)
Token 210 (,)
Token 43 (Keith)
Token 208 (moved to)
Token 105 (Manila)
Token 210 (,)
Token 18 (Lisa)
Token 200 (lives in)
Token 155 (Philadelphia)
Token 210 (,)
Token 95 (Patrick)
Token 204 (studied in)
Token 144 (Riyadh)
Token 210 (,)
Token 48 (Cynthia)
Token 202 (works in)
Token 106 (Shanghai)
Token 210 (,)
Token 37 (Christine)
Token 205 (married in)
Token 111 (Sao Paulo)
Token 210 (,)
Token 54 (Deborah)
Token 206 (visited)
Token 169 (Cape Town)
Token 210 (,)
Token 80 (Anna)
Token 207 (loves)
Token 154 (Miami)
Token 210 (,)
Token 208 (moved to)
Token 43 (Keith)
Token 211 (?)

Label: 105 (Manila)


In [43]:
# entity doesn't exist in the context
corrupt_token_2 = clean_tokens.clone()
# Find an entity ID not present in clean_tokens (assuming entity IDs are 0-99)
all_entity_ids = set(range(E))
entity_ids_in_clean_tokens = set([token_id.item() for token_id in clean_tokens if 0 <= token_id.item() < E])
new_entity_id = (list(all_entity_ids - entity_ids_in_clean_tokens) + [None])[0] # Take the first available or None

if new_entity_id is not None:
    corrupt_token_2[-2] = new_entity_id
else:
    print("Could not find an entity ID not in the clean tokens.")

print("Entity doesn't exist in the context, but not the binding:")
for token_id in corrupt_token_2:
    entity_name = id_to_entity.get(token_id.item(), f"Unknown Token {token_id.item()}")
    print(f"Token {token_id.item()} ({entity_name})")

print(f"\nLabel: {label.item()} ({id_to_entity.get(label.item(), f'Unknown Token {label.item()}')})")

Entity doesn't exist in the context, but not the binding:
Token 94 (Bryan)
Token 201 (was born in)
Token 118 (Istanbul)
Token 210 (,)
Token 43 (Keith)
Token 208 (moved to)
Token 105 (Manila)
Token 210 (,)
Token 18 (Lisa)
Token 200 (lives in)
Token 155 (Philadelphia)
Token 210 (,)
Token 95 (Patrick)
Token 204 (studied in)
Token 144 (Riyadh)
Token 210 (,)
Token 48 (Cynthia)
Token 202 (works in)
Token 106 (Shanghai)
Token 210 (,)
Token 37 (Christine)
Token 205 (married in)
Token 111 (Sao Paulo)
Token 210 (,)
Token 54 (Deborah)
Token 206 (visited)
Token 169 (Cape Town)
Token 210 (,)
Token 80 (Anna)
Token 207 (loves)
Token 154 (Miami)
Token 210 (,)
Token 208 (moved to)
Token 0 (Danielle)
Token 211 (?)

Label: 105 (Manila)


In [44]:
# entity type doesn't exist in the context
corrupt_token_3 = clean_tokens.clone()
# Find an entity type ID not present in clean_tokens (assuming entity type IDs are 200-209)
all_entity_type_ids = set(range(E + A, E + A + T))
entity_type_ids_in_clean_tokens = set([token_id.item() for token_id in clean_tokens if E + A <= token_id.item() < E + A + T])
new_entity_type_id = (list(all_entity_type_ids - entity_type_ids_in_clean_tokens) + [None])[0] # Take the first available or None

if new_entity_type_id is not None:
    corrupt_token_3[-3] = new_entity_type_id
else:
    print("Could not find an entity type ID not in the clean tokens.")

print("Entity type doesn't exist in the context, but not the binding:")
for token_id in corrupt_token_3:
    entity_name = id_to_entity.get(token_id.item(), f"Unknown Token {token_id.item()}")
    print(f"Token {token_id.item()} ({entity_name})")

print(f"\nLabel: {label.item()} ({id_to_entity.get(label.item(), f'Unknown Token {label.item()}')})")

Entity type doesn't exist in the context, but not the binding:
Token 94 (Bryan)
Token 201 (was born in)
Token 118 (Istanbul)
Token 210 (,)
Token 43 (Keith)
Token 208 (moved to)
Token 105 (Manila)
Token 210 (,)
Token 18 (Lisa)
Token 200 (lives in)
Token 155 (Philadelphia)
Token 210 (,)
Token 95 (Patrick)
Token 204 (studied in)
Token 144 (Riyadh)
Token 210 (,)
Token 48 (Cynthia)
Token 202 (works in)
Token 106 (Shanghai)
Token 210 (,)
Token 37 (Christine)
Token 205 (married in)
Token 111 (Sao Paulo)
Token 210 (,)
Token 54 (Deborah)
Token 206 (visited)
Token 169 (Cape Town)
Token 210 (,)
Token 80 (Anna)
Token 207 (loves)
Token 154 (Miami)
Token 210 (,)
Token 209 (left)
Token 43 (Keith)
Token 211 (?)

Label: 105 (Manila)


In [45]:
# neither entity and entity type exists in the context
corrupt_token_4 = clean_tokens.clone()

# Find an entity ID not present in clean_tokens (assuming entity IDs are 0-99)
all_entity_ids = set(range(E))
entity_ids_in_clean_tokens = set([token_id.item() for token_id in clean_tokens if 0 <= token_id.item() < E])
new_entity_id = (list(all_entity_ids - entity_ids_in_clean_tokens) + [None])[0]

# Find an entity type ID not present in clean_tokens (assuming entity type IDs are 200-209)
all_entity_type_ids = set(range(E + A, E + A + T))
entity_type_ids_in_clean_tokens = set([token_id.item() for token_id in clean_tokens if E + A <= token_id.item() < E + A + T])
new_entity_type_id = (list(all_entity_type_ids - entity_type_ids_in_clean_tokens) + [None])[0]

if new_entity_id is not None and new_entity_type_id is not None:
    corrupt_token_4[-2] = new_entity_id
    corrupt_token_4[-3] = new_entity_type_id
else:
    print("Could not find both a new entity and a new entity type ID not in the clean tokens.")


print("Neither entity nor entity type exist in the context:")
for token_id in corrupt_token_4:
    entity_name = id_to_entity.get(token_id.item(), f"Unknown Token {token_id.item()}")
    print(f"Token {token_id.item()} ({entity_name})")

print(f"\nLabel: {label.item()} ({id_to_entity.get(label.item(), f'Unknown Token {label.item()}')})")

Neither entity nor entity type exist in the context:
Token 94 (Bryan)
Token 201 (was born in)
Token 118 (Istanbul)
Token 210 (,)
Token 43 (Keith)
Token 208 (moved to)
Token 105 (Manila)
Token 210 (,)
Token 18 (Lisa)
Token 200 (lives in)
Token 155 (Philadelphia)
Token 210 (,)
Token 95 (Patrick)
Token 204 (studied in)
Token 144 (Riyadh)
Token 210 (,)
Token 48 (Cynthia)
Token 202 (works in)
Token 106 (Shanghai)
Token 210 (,)
Token 37 (Christine)
Token 205 (married in)
Token 111 (Sao Paulo)
Token 210 (,)
Token 54 (Deborah)
Token 206 (visited)
Token 169 (Cape Town)
Token 210 (,)
Token 80 (Anna)
Token 207 (loves)
Token 154 (Miami)
Token 210 (,)
Token 209 (left)
Token 0 (Danielle)
Token 211 (?)

Label: 105 (Manila)


In [ ]:
clean_tokens = example
# Swap each adjacent pair to get corrupted tokens
corrupted_tokens = clean_tokens[indices]

print(
    "Clean string 0:    ",
    model.to_string(clean_tokens[0]),
    "\nCorrupted string 0:",
    model.to_string(corrupted_tokens[0]),
)

clean_logits, clean_cache = model.run_with_cache(clean_tokens)
corrupted_logits, corrupted_cache = model.run_with_cache(corrupted_tokens)

clean_logit_diff = logits_to_ave_logit_diff(clean_logits, answer_tokens)
print(f"Clean logit diff: {clean_logit_diff:.4f}")

corrupted_logit_diff = logits_to_ave_logit_diff(corrupted_logits, answer_tokens)
print(f"Corrupted logit diff: {corrupted_logit_diff:.4f}")